# Experiment 3 Postprocessing: Redundancy Elimination

This notebook generates tables and figures for cache-hit and dedup behavior across the 3 staged runs:
- `phase_a_full_100`
- `phase_b_warmup_80`
- `phase_c_replay_100`

Input source is folder-first (`outputs/source_exports/*`), with manifest used only as optional metadata.


In [11]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import re
from datetime import datetime
from typing import Any

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11
})

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'orchestrator.py').exists() and (candidate / 'experiments').is_dir():
            return candidate
    return None


ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    raise RuntimeError(f'Could not locate fl-net-sim repository root from {Path.cwd()}')

EXP_ID = 'exp3_redundancy_elimination'
EXP_DIR = ROOT / 'experiments' / EXP_ID
OUTPUTS_DIR = EXP_DIR / 'outputs'
TABLES_DIR = OUTPUTS_DIR / 'tables'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
SOURCE_EXPORTS_DIR = OUTPUTS_DIR / 'source_exports'
MANIFEST_PATH = OUTPUTS_DIR / 'last_run_manifest.json'

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ORDER = [
    'phase_a_full_100',
    'phase_b_warmup_80',
    'phase_c_replay_100',
]

PHASE_LABEL = {
    'phase_a_full_100': 'A: Cold full 1-100',
    'phase_b_warmup_80': 'B: Warmup 1-80',
    'phase_c_replay_100': 'C: Replay 1-100',
}

print(f'ROOT={ROOT}')
print(f'EXP_DIR={EXP_DIR}')
print(f'SOURCE_EXPORTS_DIR={SOURCE_EXPORTS_DIR}')


ROOT=/mnt/nas_drive/psklavos/MDM2026/fl-net-sim
EXP_DIR=/mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination
SOURCE_EXPORTS_DIR=/mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/source_exports


In [7]:
def to_markdown_no_deps(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        safe = df.fillna('')
        cols = [str(c) for c in safe.columns]
        header = '| ' + ' | '.join(cols) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
        rows = [
            '| ' + ' | '.join(str(v).replace('\n', ' ').replace('|', '\\|') for v in row) + ' |'
            for row in safe.itertuples(index=False, name=None)
        ]
        return '\n'.join([header, sep, *rows])


def load_round_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    text = path.read_text(encoding='utf-8', errors='ignore')
    section = text.split('\n\n', 1)[0].strip()
    if not section:
        return pd.DataFrame()
    rows = list(csv.DictReader(section.splitlines()))
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    for col in frame.columns:
        converted = pd.to_numeric(frame[col], errors='coerce')
        if converted.notna().any():
            frame[col] = converted
    return frame


NETWORK_RE = re.compile(
    r'network_type=\w+\s+rounds=(?P<rounds>\d+)\s+parallelism=(?P<parallelism>\d+)\s+cached=(?P<cached>\d+)\s+deduped=(?P<deduped>\d+)'
)
PLANNED_RE = re.compile(r'planned\s+(?P<planned>\d+)\s+rounds:')
TS_RE = re.compile(r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:,\d+)?)\s+\|\s+INFO\s+\|')
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S,%f']


def parse_log_metrics(log_path: Path) -> dict[str, Any]:
    out = {
        'rounds_requested': 0,
        'planned_rounds': 0,
        'cached_rounds': 0,
        'deduped_rounds': 0,
        'parallelism_reported': 0,
        'exec_time_from_log_s': np.nan,
    }
    if not log_path.exists():
        return out

    text = log_path.read_text(encoding='utf-8', errors='ignore')

    network = NETWORK_RE.findall(text)
    if network:
        rounds, parallelism, cached, deduped = network[-1]
        out['rounds_requested'] = int(rounds)
        out['parallelism_reported'] = int(parallelism)
        out['cached_rounds'] = int(cached)
        out['deduped_rounds'] = int(deduped)

    planned = PLANNED_RE.findall(text)
    if planned:
        out['planned_rounds'] = int(planned[-1])

    first_ts = None
    last_ts = None
    for line in text.splitlines():
        m = TS_RE.match(line)
        if not m:
            continue
        ts_text = m.group('ts')
        ts = None
        for fmt in TS_FORMATS:
            try:
                ts = datetime.strptime(ts_text, fmt)
                break
            except ValueError:
                continue
        if ts is None:
            continue
        if first_ts is None:
            first_ts = ts
        last_ts = ts

    if first_ts is not None and last_ts is not None and last_ts >= first_ts:
        out['exec_time_from_log_s'] = (last_ts - first_ts).total_seconds()

    return out


In [8]:
manifest_runs_by_phase: dict[str, dict[str, Any]] = {}
if MANIFEST_PATH.exists():
    try:
        manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        for item in manifest.get('runs', []):
            phase_id = str(item.get('phase_id', '')).strip()
            if phase_id:
                manifest_runs_by_phase[phase_id] = item
        print(f'Loaded manifest metadata: {MANIFEST_PATH} (runs={len(manifest_runs_by_phase)})')
    except Exception as exc:
        print(f'Warning: failed to parse manifest {MANIFEST_PATH}: {exc}')
else:
    print(f'No manifest found at {MANIFEST_PATH}; using folder-only discovery.')

if not SOURCE_EXPORTS_DIR.exists():
    raise FileNotFoundError(f'Missing source exports directory: {SOURCE_EXPORTS_DIR}')

phase_dirs = [p for p in SOURCE_EXPORTS_DIR.iterdir() if p.is_dir()]
phase_ids_found = sorted(p.name for p in phase_dirs)

phase_ids: list[str] = [p for p in PHASE_ORDER if p in phase_ids_found]
for p in phase_ids_found:
    if p not in phase_ids:
        phase_ids.append(p)

if not phase_ids:
    raise FileNotFoundError(f'No phase directories found under: {SOURCE_EXPORTS_DIR}')

phase_records: list[dict[str, Any]] = []
requested_tables: dict[str, pd.DataFrame] = {}
round_index_tables: dict[str, pd.DataFrame] = {}

for phase_id in phase_ids:
    phase_dir = SOURCE_EXPORTS_DIR / phase_id
    manifest_item = manifest_runs_by_phase.get(phase_id, {})

    log_path = phase_dir / 'orchestrator_capture.log'
    if not log_path.exists() and manifest_item.get('capture_log'):
        log_path = Path(manifest_item['capture_log'])

    requested_csv = phase_dir / 'requested_rounds.csv'
    if not requested_csv.exists() and manifest_item.get('copied_requested_rounds_csv'):
        requested_csv = Path(manifest_item['copied_requested_rounds_csv'])

    round_index_csv = phase_dir / 'round_index.csv'
    if not round_index_csv.exists() and manifest_item.get('copied_round_index_csv'):
        round_index_csv = Path(manifest_item['copied_round_index_csv'])

    requested_df = load_round_table(requested_csv)
    round_index_df = load_round_table(round_index_csv)
    requested_tables[phase_id] = requested_df
    round_index_tables[phase_id] = round_index_df

    parsed = parse_log_metrics(log_path)

    planned_rounds = int(parsed['planned_rounds'])
    if planned_rounds == 0 and not requested_df.empty:
        planned_rounds = int(len(requested_df))

    rounds_requested = int(parsed['rounds_requested'])
    if rounds_requested == 0:
        rounds_requested = planned_rounds

    exec_time_s = manifest_item.get('exec_time_s', np.nan)
    exec_time_s = pd.to_numeric(pd.Series([exec_time_s]), errors='coerce').iloc[0]
    if pd.isna(exec_time_s):
        exec_time_s = parsed['exec_time_from_log_s']

    cached_rounds = int(parsed['cached_rounds'])
    deduped_rounds = int(parsed['deduped_rounds'])
    parallelism_reported = int(parsed['parallelism_reported'])

    phase_records.append(
        {
            'phase_id': phase_id,
            'phase_label': PHASE_LABEL.get(phase_id, phase_id),
            'exports_dir': str(phase_dir),
            'log_path': str(log_path),
            'requested_rounds_csv': str(requested_csv) if requested_csv.exists() else '',
            'round_index_csv': str(round_index_csv) if round_index_csv.exists() else '',
            'exec_time_s': float(exec_time_s) if pd.notna(exec_time_s) else np.nan,
            'rounds_requested': rounds_requested,
            'planned_rounds': planned_rounds,
            'cached_rounds': cached_rounds,
            'deduped_rounds': deduped_rounds,
            'parallelism_reported': parallelism_reported,
        }
    )

phase_summary = pd.DataFrame(phase_records)
if phase_summary.empty:
    raise ValueError('No phase data discovered for exp3 postprocessing.')

phase_summary['phase_id'] = pd.Categorical(phase_summary['phase_id'], categories=phase_ids, ordered=True)
phase_summary = phase_summary.sort_values('phase_id').reset_index(drop=True)
phase_summary['phase_id'] = phase_summary['phase_id'].astype(str)

phase_summary['cache_hit_ratio'] = phase_summary['cached_rounds'] / phase_summary['rounds_requested'].clip(lower=1)
phase_summary['dedup_ratio'] = phase_summary['deduped_rounds'] / phase_summary['rounds_requested'].clip(lower=1)
phase_summary['elimination_ratio'] = (phase_summary['cached_rounds'] + phase_summary['deduped_rounds']) / phase_summary['rounds_requested'].clip(lower=1)
phase_summary['executed_unique_rounds'] = (
    phase_summary['rounds_requested'] - phase_summary['cached_rounds'] - phase_summary['deduped_rounds']
).clip(lower=0)
phase_summary['avg_time_per_requested_round_s'] = phase_summary['exec_time_s'] / phase_summary['rounds_requested'].clip(lower=1)

phase_summary_out = phase_summary[
    [
        'phase_id',
        'phase_label',
        'exec_time_s',
        'avg_time_per_requested_round_s',
        'rounds_requested',
        'planned_rounds',
        'cached_rounds',
        'deduped_rounds',
        'executed_unique_rounds',
        'cache_hit_ratio',
        'dedup_ratio',
        'elimination_ratio',
        'parallelism_reported',
    ]
].copy()

csv_path = TABLES_DIR / 'exp3_phase_summary.csv'
md_path = TABLES_DIR / 'exp3_phase_summary.md'
phase_summary_out.to_csv(csv_path, index=False)
md_path.write_text(to_markdown_no_deps(phase_summary_out), encoding='utf-8')

print(f'Wrote {csv_path}')
print(f'Wrote {md_path}')
phase_summary_out


Loaded manifest metadata: /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/last_run_manifest.json (runs=3)
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/tables/exp3_phase_summary.csv
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/tables/exp3_phase_summary.md


,phase_id,phase_label,exec_time_s,avg_time_per_requested_round_s,rounds_requested,planned_rounds,cached_rounds,deduped_rounds,executed_unique_rounds,cache_hit_ratio,dedup_ratio,elimination_ratio,parallelism_reported
0,phase_a_full_100,A: Cold full 1-100,64.472623,0.644726,100,100,0,0,100,0.00,0.0,0.00,20
1,phase_b_warmup_80,B: Warmup 1-80,45.619574,0.570245,80,80,0,16,64,0.00,0.2,0.20,20
2,phase_c_replay_100,C: Replay 1-100,12.411266,0.124113,100,100,89,0,11,0.89,0.0,0.89,20


In [9]:
warmup_df = requested_tables.get('phase_b_warmup_80', pd.DataFrame()).copy()
replay_df = requested_tables.get('phase_c_replay_100', pd.DataFrame()).copy()

if replay_df.empty:
    raise ValueError('Missing replay table for phase_c_replay_100. Run experiment first.')

for required_col in ['index', 'round', 'round_key']:
    if required_col not in replay_df.columns:
        raise ValueError(f'Missing required column {required_col!r} in replay requested_rounds.csv')

for c in ['index', 'round']:
    replay_df[c] = pd.to_numeric(replay_df[c], errors='coerce')
replay_df = replay_df.dropna(subset=['index', 'round']).copy()
replay_df['index'] = replay_df['index'].astype(int)
replay_df['round'] = replay_df['round'].astype(int)
replay_df['round_key'] = replay_df['round_key'].astype(str)

if not warmup_df.empty:
    if 'index' in warmup_df.columns:
        warmup_df['index'] = pd.to_numeric(warmup_df['index'], errors='coerce')
    if 'round_key' in warmup_df.columns:
        warmup_df['round_key'] = warmup_df['round_key'].astype(str)

warmup_requested_set: set[int] = set()
if 'index' in warmup_df.columns:
    warmup_requested_set = set(warmup_df['index'].dropna().astype(int).tolist())

warmup_key_set: set[str] = set()
if 'round_key' in warmup_df.columns:
    warmup_key_set = set(warmup_df['round_key'].dropna().astype(str).tolist())

# Primary target: replay tail rounds 81..100.
tail_df = replay_df[(replay_df['index'] >= 81) & (replay_df['index'] <= 100)].copy()

# Fallback if user ran with different round count: use last 20 requested rounds.
if tail_df.empty:
    max_idx = int(replay_df['index'].max())
    tail_df = replay_df[replay_df['index'] > max_idx - 20].copy()

tail_df = tail_df.sort_values('index').reset_index(drop=True)

tail_df['is_cached_from_warmup'] = tail_df['index'].isin(warmup_requested_set)
tail_df['is_deduped'] = tail_df['round'] != tail_df['index']
tail_df['dedup_hits_warmup_key'] = tail_df['round_key'].isin(warmup_key_set)

def classify_row(row: pd.Series) -> str:
    if bool(row['is_cached_from_warmup']):
        return 'cached_from_warmup'
    if bool(row['is_deduped']) and bool(row['dedup_hits_warmup_key']):
        return 'dedup_to_warmup'
    if bool(row['is_deduped']):
        return 'dedup_within_tail_or_unknown'
    return 'executed_unique'

tail_df['tail_status'] = tail_df.apply(classify_row, axis=1)

cached_in_tail = int((tail_df['tail_status'] == 'cached_from_warmup').sum())
deduped_in_tail = int(tail_df['tail_status'].isin(['dedup_to_warmup', 'dedup_within_tail_or_unknown']).sum())
executed_unique_in_tail = int((tail_df['tail_status'] == 'executed_unique').sum())

tail_n = int(len(tail_df))

replay_row = phase_summary_out[phase_summary_out['phase_id'] == 'phase_c_replay_100']
if replay_row.empty:
    replay_row = phase_summary_out.iloc[[-1]]

overall_cache_hit_ratio = float(replay_row['cache_hit_ratio'].iloc[0])
overall_dedup_ratio = float(replay_row['dedup_ratio'].iloc[0])
overall_elimination_ratio = float(replay_row['elimination_ratio'].iloc[0])

last20_summary = pd.DataFrame(
    [
        {
            'phase_id': 'phase_c_replay_100',
            'tail_start_requested_round': int(tail_df['index'].min()) if tail_n else np.nan,
            'tail_end_requested_round': int(tail_df['index'].max()) if tail_n else np.nan,
            'tail_round_count': tail_n,
            'cached_from_warmup_count': cached_in_tail,
            'deduped_in_tail_count': deduped_in_tail,
            'executed_unique_in_tail_count': executed_unique_in_tail,
            'cached_from_warmup_ratio': (cached_in_tail / tail_n) if tail_n else np.nan,
            'deduped_in_tail_ratio': (deduped_in_tail / tail_n) if tail_n else np.nan,
            'executed_unique_in_tail_ratio': (executed_unique_in_tail / tail_n) if tail_n else np.nan,
            'overall_replay_cache_hit_ratio': overall_cache_hit_ratio,
            'overall_replay_dedup_ratio': overall_dedup_ratio,
            'overall_replay_elimination_ratio': overall_elimination_ratio,
        }
    ]
)

last20_details = tail_df[
    [
        'index',
        'round',
        'round_key',
        'tail_status',
        'is_cached_from_warmup',
        'is_deduped',
        'dedup_hits_warmup_key',
    ]
].rename(
    columns={
        'index': 'requested_round',
        'round': 'resolved_round',
    }
)

summary_csv = TABLES_DIR / 'exp3_replay_last20_summary.csv'
summary_md = TABLES_DIR / 'exp3_replay_last20_summary.md'
details_csv = TABLES_DIR / 'exp3_replay_last20_details.csv'

last20_summary.to_csv(summary_csv, index=False)
summary_md.write_text(to_markdown_no_deps(last20_summary), encoding='utf-8')
last20_details.to_csv(details_csv, index=False)

print(f'Wrote {summary_csv}')
print(f'Wrote {summary_md}')
print(f'Wrote {details_csv}')

last20_summary


Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/tables/exp3_replay_last20_summary.csv
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/tables/exp3_replay_last20_summary.md
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/tables/exp3_replay_last20_details.csv


,phase_id,tail_start_requested_round,tail_end_requested_round,tail_round_count,cached_from_warmup_count,deduped_in_tail_count,executed_unique_in_tail_count,cached_from_warmup_ratio,deduped_in_tail_ratio,executed_unique_in_tail_ratio,overall_replay_cache_hit_ratio,overall_replay_dedup_ratio,overall_replay_elimination_ratio
0,phase_c_replay_100,81,100,20,0,9,11,0.0,0.45,0.55,0.89,0.0,0.89


In [10]:
phase_for_fig = phase_summary_out.copy()
phase_for_fig['phase_label'] = pd.Categorical(
    phase_for_fig['phase_label'],
    categories=[PHASE_LABEL.get(p, p) for p in phase_ids],
    ordered=True,
)
phase_for_fig = phase_for_fig.sort_values('phase_label').reset_index(drop=True)

fig1 = FIGURES_DIR / 'exp3_phase_cache_dedup_ratios.png'
fig2 = FIGURES_DIR / 'exp3_phase_exec_time.png'
fig3 = FIGURES_DIR / 'exp3_replay_last20_breakdown.png'

x = np.arange(len(phase_for_fig))
width = 0.26

plt.figure(figsize=(11, 5))
plt.bar(x - width, phase_for_fig['cache_hit_ratio'], width=width, label='Cache hit ratio')
plt.bar(x, phase_for_fig['dedup_ratio'], width=width, label='Dedup ratio')
plt.bar(x + width, phase_for_fig['elimination_ratio'], width=width, label='Total elimination ratio')
plt.xticks(x, phase_for_fig['phase_label'], rotation=10, ha='right')
plt.ylim(0, 1.0)
plt.ylabel('Ratio')
plt.title('Experiment 3: Cache and Dedup Ratios by Phase')
plt.legend()
plt.tight_layout()
plt.savefig(fig1, dpi=180)
plt.close()

plt.figure(figsize=(10, 5))
plt.bar(phase_for_fig['phase_label'], phase_for_fig['exec_time_s'])
plt.xticks(rotation=10, ha='right')
plt.ylabel('Execution time (s)')
plt.title('Experiment 3: Execution Time by Phase')
plt.tight_layout()
plt.savefig(fig2, dpi=180)
plt.close()

breakdown = {
    'cached_from_warmup': int(last20_summary['cached_from_warmup_count'].iloc[0]),
    'deduped_in_tail': int(last20_summary['deduped_in_tail_count'].iloc[0]),
    'executed_unique_in_tail': int(last20_summary['executed_unique_in_tail_count'].iloc[0]),
}

plt.figure(figsize=(8, 5))
plt.bar(list(breakdown.keys()), list(breakdown.values()))
plt.ylabel('Round count')
plt.title('Experiment 3 Replay Tail (Last 20): Breakdown')
plt.tight_layout()
plt.savefig(fig3, dpi=180)
plt.close()

print(f'Wrote {fig1}')
print(f'Wrote {fig2}')
print(f'Wrote {fig3}')


Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/figures/exp3_phase_cache_dedup_ratios.png
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/figures/exp3_phase_exec_time.png
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp3_redundancy_elimination/outputs/figures/exp3_replay_last20_breakdown.png
